# 🎙️ Wav2Lip Studio - Khớp Khẩu Hình Video (Tự Động Lưu Vào Google Drive)
> **Hướng dẫn 1-Click (Không cần tải lại lần sau):**
> 1. Bấm nút **'Sao chép vào Drive'** ở thanh trên cùng để lưu vĩnh viễn notebook này vào Google Drive của bạn.
> 2. Bấm **Runtime (Thời gian chạy)** -> **Change runtime type** -> Chọn **T4 GPU**.
> 3. Bấm **Runtime** -> **Run all (Chạy tất cả)**.
> 4. Khi Colab hỏi quyền truy cập Google Drive, chọn **'Kết nối với Google Drive'** (Connect to Google Drive). Lần đầu sẽ tải và lưu model vào Drive, **từ lần thứ 2 trở đi sẽ load tức thì từ Drive mà KHÔNG CẦN TẢI LẠI!**

In [ ]:
#@title 1. Kết nối Google Drive & Kiểm tra Caching Model
import os
import shutil
from google.colab import drive
from IPython.display import clear_output

# 1. Gắn kết Google Drive
print("🔗 Đang kết nối với Google Drive của bạn...")
drive.mount('/content/drive')

drive_cache_dir = "/content/drive/MyDrive/AI_Colab_Cache/Wav2Lip"
os.makedirs("/content/drive/MyDrive/AI_Colab_Cache", exist_ok=True)

# 2. Kiểm tra nếu đã có sẵn trên Drive thì dùng luôn, chưa có thì tải 1 lần duy nhất
if os.path.exists(drive_cache_dir) and os.path.exists(f"{drive_cache_dir}/checkpoints/wav2lip_gan.pth"):
    print("🎉 ĐÃ TÌM THẤY MODEL TRONG GOOGLE DRIVE! Bỏ qua tải về, nạp trực tiếp...")
    if not os.path.exists("/content/Wav2Lip"):
        !cp -r "{drive_cache_dir}" /content/Wav2Lip
else:
    print("⏳ Chưa có trong Drive. Đang tải model lần đầu và lưu vào Google Drive của bạn...")
    !git clone https://huggingface.co/camenduru/Wav2Lip /content/Wav2Lip
    print("💾 Đang lưu bản sao model vào Google Drive để lần sau không phải tải lại...")
    !cp -r /content/Wav2Lip "{drive_cache_dir}"

%cd /content/Wav2Lip
!pip install -q gradio==3.50.2 yt_dlp ffmpeg-python librosa==0.8.0
clear_output()
print("✅ Môi trường đã sẵn sàng! Đang khởi chạy WebUI...")


In [ ]:
#@title 2. Khởi chạy Giao diện WebUI Khớp Khẩu Hình
import gradio as gr
import os
import shutil
from yt_dlp import YoutubeDL

os.makedirs("/content/Wav2Lip/results", exist_ok=True)
output_drive_dir = "/content/drive/MyDrive/AI_Colab_Cache/Wav2Lip_Outputs"
os.makedirs(output_drive_dir, exist_ok=True)

def sync_lips(video_file, audio_file, youtube_url, pads_top, pads_bottom, pads_left, pads_right):
    target_video = "/content/input_video.mp4"
    if video_file is not None:
        shutil.copy(video_file, target_video)
    elif youtube_url and len(youtube_url.strip()) > 5:
        ydl_opts = {"overwrites": True, "format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/mp4", "outtmpl": target_video}
        with YoutubeDL(ydl_opts) as ydl:
            ydl.download(youtube_url)
    else:
        return None, "❌ Vui lòng tải lên video hoặc dán link YouTube!"
    
    if audio_file is None:
        return None, "❌ Vui lòng tải lên file âm thanh tiếng Việt!"
    
    output_path = "/content/Wav2Lip/results/result_voice.mp4"
    cmd = f"python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth --face '{target_video}' --audio '{audio_file}' --pads {pads_top} {pads_bottom} {pads_left} {pads_right} --outfile '{output_path}'"
    os.system(cmd)
    
    if os.path.exists(output_path):
        # Tự động lưu 1 bản vào Google Drive để không bị mất khi đóng Colab
        import time
        ts_name = f"lipsync_{int(time.time())}.mp4"
        shutil.copy(output_path, f"{output_drive_dir}/{ts_name}")
        return output_path, f"🎉 Khớp khẩu hình thành công! Đã tự động lưu 1 bản vào Google Drive ({ts_name})."
    else:
        return None, "❌ Có lỗi trong quá trình xử lý. Hãy kiểm tra lại định dạng video/âm thanh!"

with gr.Blocks(title="Wav2Lip Cloud Studio") as demo:
    gr.Markdown("## 🎬 Wav2Lip AI - Khớp Khẩu Hình Siêu Tốc (Tesla T4 Cloud GPU)")
    gr.Markdown("Tải lên video gốc của bạn và file âm thanh lồng tiếng tiếng Việt để AI tự động đồng bộ cử động môi theo nhịp nói.")
    with gr.Row():
        with gr.Column():
            video_input = gr.Video(label="1. Tải lên Video gốc (MP4)", type="filepath")
            yt_input = gr.Textbox(label="Hoặc dán Link YouTube nếu không tải video lên", placeholder="https://youtu.be/...")
            audio_input = gr.Audio(label="2. Tải lên File Âm thanh Tiếng Việt (MP3 / WAV)", type="filepath")
            with gr.Accordion("Tùy chỉnh đệm viền môi (Pads) - Mặc định chuẩn", open=False):
                p_top = gr.Slider(0, 20, value=0, step=1, label="Pads Top")
                p_bottom = gr.Slider(0, 20, value=10, step=1, label="Pads Bottom (Đệm cằm)")
                p_left = gr.Slider(0, 20, value=0, step=1, label="Pads Left")
                p_right = gr.Slider(0, 20, value=0, step=1, label="Pads Right")
            btn_run = gr.Button("🚀 Bắt Đầu Khớp Khẩu Hình (Generate Lip-Sync)", variant="primary")
        with gr.Column():
            video_output = gr.Video(label="Video Kết Quả (Đã Khớp Môi)")
            status_msg = gr.Textbox(label="Trạng thái xử lý", interactive=False)
    btn_run.click(sync_lips, inputs=[video_input, audio_input, yt_input, p_top, p_bottom, p_left, p_right], outputs=[video_output, status_msg])

demo.queue().launch(share=True, debug=False)
